<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session06-discussion-3.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 6 — In-class discussion problem (3 of 3)

Discuss **as a group first**, write down a guess, then run the code to
check it before presenting.

## Can one distance summarize a conformational change?

The main notebook measured *one* number, the distance between the iron
atoms of the two β-chain hemes, and found it about 5 Å shorter in oxy
(R, `1HHO`) than in deoxy (T, `2HHB`) hemoglobin. But a change of shape
involves thousands of atoms. Can one distance summarize it?

Hemoglobin's tetramer has four distinct chain–chain pairs: α1–β1, α1–β2,
α1–α2 and β1–β2. Structural biologists (Perutz; Baldwin & Chothia, 1979)
call α1–β1 the "fixed" interface and α1–β2 the "sliding" one, where the
two α/β pairs move against each other during the transition.

**As a group, before running anything:**

1. Which of the four iron–iron distances do you expect to change the
   **most**, and which the **least**?
2. Do the individual chains change shape between the two states, or do
   they move as rigid bodies?
3. If the two α/β pairs rotate against each other, would every
   iron–iron distance necessarily change a lot?

Then run the cells below. The first computes the four distances in both
states (the R-state file contains only one α/β pair; the other is
generated with the file's own `REMARK 350` symmetry operation). The
second superimposes the structures.

In [1]:
# Install the libraries Colab does not come with (does nothing if they are already installed)
import importlib.util, subprocess, sys
for module, package in [("Bio", "biopython")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

In [2]:
import numpy as np
from Bio.PDB import PDBList, PDBParser

pdbl = PDBList()
paths = {
    "2hhb": pdbl.retrieve_pdb_file("2hhb", pdir=".", file_format="pdb"),
    "1hho": pdbl.retrieve_pdb_file("1hho", pdir=".", file_format="pdb"),
}

parser = PDBParser(QUIET=True)
structures = {pdb_id: parser.get_structure(pdb_id, paths[pdb_id]) for pdb_id in paths}

# --- T-state (2HHB): all four chains already present in the file ---
fe_T = {}
for chain in structures["2hhb"][0]:
    for res in chain:
        if res.get_resname() == "HEM":
            fe_T[chain.id] = np.array(res["FE"].get_coord(), dtype=float)
# A, C = alpha chains; B, D = beta chains

# --- R-state (1HHO): only one alpha/beta pair in the file; the other
#     alpha/beta pair is generated by the file's own BIOMT symmetry
#     operation (swap x/y, negate z, no translation) ---
fe_R = {}
for chain in structures["1hho"][0]:
    for res in chain:
        if res.get_resname() == "HEM":
            fe_R[chain.id] = np.array(res["FE"].get_coord(), dtype=float)
Rmat = np.array([[0, 1, 0], [1, 0, 0], [0, 0, -1]])
fe_R["Ap"] = Rmat @ fe_R["A"]
fe_R["Bp"] = Rmat @ fe_R["B"]
# A, Ap = alpha chains; B, Bp = beta chains

def dist(fe, a, b):
    return np.linalg.norm(fe[a] - fe[b])

pairs_T = {
    "alpha1-beta1": (dist(fe_T, "A", "B") + dist(fe_T, "C", "D")) / 2,
    "alpha1-beta2": (dist(fe_T, "A", "D") + dist(fe_T, "C", "B")) / 2,
    "alpha1-alpha2": dist(fe_T, "A", "C"),
    "beta1-beta2": dist(fe_T, "B", "D"),
}
pairs_R = {
    "alpha1-beta1": dist(fe_R, "A", "B"),
    "alpha1-beta2": dist(fe_R, "A", "Bp"),
    "alpha1-alpha2": dist(fe_R, "A", "Ap"),
    "beta1-beta2": dist(fe_R, "B", "Bp"),
}

print(f"{'interface':16s} {'T-state':>10s} {'R-state':>10s} {'change':>10s}")
for name in pairs_T:
    t, r = pairs_T[name], pairs_R[name]
    print(f"{name:16s} {t:9.2f}A {r:9.2f}A {t - r:+9.2f}A")

Structure exists: './pdb2hhb.ent' 
Structure exists: './pdb1hho.ent' 
interface           T-state    R-state     change
alpha1-beta1         36.46A     34.73A     +1.73A
alpha1-beta2         24.30A     25.50A     -1.19A
alpha1-alpha2        34.16A     34.75A     -0.59A
beta1-beta2          39.51A     34.62A     +4.89A


In [3]:
# Superposition: how much does each chain change, and how do the two alpha/beta pairs move?
def ca(chain):
    return {r.id[1]: np.array(r["CA"].coord, dtype=float) for r in chain if r.id[0] == " " and "CA" in r}

def matched(a, b):
    keys = sorted(set(a) & set(b))
    return np.array([a[k] for k in keys]), np.array([b[k] for k in keys])

def superpose(P, Q):
    """Rotation R and translation t that best map points P onto Q (Kabsch), and the RMSD after fitting."""
    Pc, Qc = P.mean(0), Q.mean(0)
    U, S, Vt = np.linalg.svd((P - Pc).T @ (Q - Qc))
    D = np.diag([1, 1, np.sign(np.linalg.det(Vt.T @ U.T))])
    R = Vt.T @ D @ U.T
    t = Qc - Pc @ R.T
    return R, t, np.sqrt((((P @ R.T + t) - Q) ** 2).sum(1).mean())

T, Rst = structures["2hhb"][0], structures["1hho"][0]
t_ca = {c: ca(T[c]) for c in "ABCD"}
r_ca = {"A": ca(Rst["A"]), "B": ca(Rst["B"])}
r_ca["C"] = {k: Rmat @ v for k, v in r_ca["A"].items()}     # second alpha/beta pair from REMARK 350
r_ca["D"] = {k: Rmat @ v for k, v in r_ca["B"].items()}

print("Each chain superimposed on its own counterpart (C-alpha RMSD):")
for c, name in zip("AB", ["alpha", "beta"]):
    print(f"  {name} chain: {superpose(*matched(t_ca[c], r_ca[c]))[2]:.2f} A")

# Superimpose the first alpha/beta pair (A+B) of T onto R, then look at the second pair (C+D)
P1 = np.vstack([matched(t_ca[c], r_ca[c])[0] for c in "AB"])
Q1 = np.vstack([matched(t_ca[c], r_ca[c])[1] for c in "AB"])
R1, t1, rmsd1 = superpose(P1, Q1)
P2 = np.vstack([matched(t_ca[c], r_ca[c])[0] for c in "CD"]) @ R1.T + t1
Q2 = np.vstack([matched(t_ca[c], r_ca[c])[1] for c in "CD"])
R2, _, _ = superpose(P2, Q2)
angle = np.degrees(np.arccos((np.trace(R2) - 1) / 2))
print(f"\nFirst alpha/beta pair superimposed: RMSD {rmsd1:.2f} A")
print(f"Second alpha/beta pair, without refitting: RMSD {np.sqrt(((P2 - Q2) ** 2).sum(1).mean()):.1f} A")
print(f"Rotation that brings the second pair into place: {angle:.0f} degrees")

Each chain superimposed on its own counterpart (C-alpha RMSD):
  alpha chain: 0.70 A
  beta chain: 0.82 A

First alpha/beta pair superimposed: RMSD 0.94 A
Second alpha/beta pair, without refitting: RMSD 5.1 A
Rotation that brings the second pair into place: 13 degrees


**Discussion points.**

*The four distances* tell different stories. β1–β2 changes most (about
5 Å), α1–α2 barely at all (about 0.6 Å), and the "sliding" α1–β2 pair
(about 1.2 Å) less than the "fixed" α1–β1 pair (about 1.7 Å). Picking any
one of them would give a different impression of how much the protein
changed, and none of them tells you *how* it moved.

*The superposition* shows what actually happens. Each chain on its own
hardly changes (C-alpha RMSD below 1 Å). With the first α/β pair
superimposed, the second pair is off by about 5 Å and must be rotated by
about 13° to fit: the two α/β pairs move against each other as rigid
bodies. A rotation moves atoms by different amounts depending on how far
they are from the rotation axis, which is why the iron–iron distances
change by such different amounts.

*So, can one distance summarize a conformational change?* No. A single
distance is one projection of a motion of thousands of atoms. Describing
the change needs the whole picture: which parts move as rigid units, the
rotation and translation between them, and which interfaces change. The
broader lesson for later sessions: a PDB entry is one state of a
protein, and comparing a prediction with "the" structure needs to ask
*which* state. One group presents.